# 🤖 Colab LLM Server → Hermes (Ollama + Cloudflare Tunnel)

يشغّل **Qwen2.5-Coder-7B** على GPU مجاني بواجهة متوافقة مع OpenAI،
محمي بمفتاح، ومعروض على `https://colab-llm.cloudstars.club/v1` عبر Cloudflare Tunnel ثابت.

## قبل التشغيل (مرة واحدة):
1. **Runtime → Change runtime type → T4 GPU** ثم Save.
2. أيقونة المفتاح 🔑 (**Secrets**) → ضيف وفعّل **Notebook access** لكل واحد:
   - `CF_TUNNEL_TOKEN` = توكن تونل `colab-llm` (يبدأ بـ `eyJ...`).
   - `LLM_API_KEY` = أي مفتاح سرّي تختاره (لحماية الـ endpoint). لو فاضي هيتولّد تلقائي.
3. **Runtime → Run all**.

> اترك التبويب مفتوح؛ خمول ~90 دقيقة يوقف الجلسة.


### 1️⃣ تأكد من الـ GPU


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "⚠️ مفيش GPU! Runtime > Change runtime type > T4 GPU"
print("✅ GPU:", torch.cuda.get_device_name(0))

### 2️⃣ تثبيت Ollama + cloudflared (~1-2 دقيقة)


In [ ]:
# zstd مطلوبة لفك ضغط Ollama على Colab
!apt-get -qq install -y zstd

# Ollama (محرك تشغيل مستقل — بلا تعارضات CUDA)
!curl -fsSL https://ollama.com/install.sh | sh

# cloudflared (النفق الثابت)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print("✅ تم التثبيت")

### 3️⃣ الإعدادات والمفاتيح


In [ ]:
from google.colab import userdata
import secrets

MODEL = "qwen2.5-coder:7b"   # الاسم اللي هتستخدمه في Hermes كمان
PORT  = 8000                  # بورت البروكسي المحمي (اللي التونل بيوصّله)

CF_TUNNEL_TOKEN = userdata.get('CF_TUNNEL_TOKEN')
try:
    LLM_API_KEY = userdata.get('LLM_API_KEY')
except Exception:
    LLM_API_KEY = None
if not LLM_API_KEY:
    LLM_API_KEY = "sk-colab-" + secrets.token_hex(16)

assert CF_TUNNEL_TOKEN, "⚠️ ضيف CF_TUNNEL_TOKEN في Colab Secrets (🔑)"
print("✅ MODEL       =", MODEL)
print("🔑 LLM_API_KEY =", LLM_API_KEY, "  ← حطّه في إعدادات Hermes")

### 4️⃣ تشغيل Ollama وتنزيل الموديل (أول مرة ~3-5 دقائق)


In [ ]:
import subprocess, os, time, requests
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

ol_log = open("/content/ollama.log", "w")
ol_proc = subprocess.Popen(["ollama", "serve"], stdout=ol_log, stderr=subprocess.STDOUT, env=os.environ.copy())

up = False
for i in range(60):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=3).status_code == 200:
            up = True; break
    except Exception:
        pass
    time.sleep(2)
print("✅ ollama serve شغّال" if up else "❌ ollama مبدأش — شوف /content/ollama.log")

print("⏳ بنزّل", MODEL, "...")
subprocess.run(["ollama", "pull", MODEL])
tags = requests.get("http://127.0.0.1:11434/api/tags").json()
print("✅ الموديلات المتاحة:", [m["name"] for m in tags.get("models", [])])

### 5️⃣ بروكسي محمي بمفتاح على البورت 8000


In [ ]:
import threading, time
from flask import Flask, request, Response
import requests as rq

OLLAMA = "http://127.0.0.1:11434"
app = Flask(__name__)

@app.route('/_diag')
def _diag():
    if request.args.get('k') != LLM_API_KEY:
        return Response('forbidden', 403)
    out = ""
    for f in ['/content/ollama.log', '/content/cloudflared.log']:
        try:
            out += f"\n==== {f} ====\n" + open(f).read()[-3000:]
        except Exception as e:
            out += f"\n{f}: {e}"
    return Response(out, mimetype='text/plain')

@app.route('/', defaults={'p': ''}, methods=['GET', 'POST'])
@app.route('/<path:p>', methods=['GET', 'POST'])
def proxy(p):
    if request.headers.get('Authorization', '') != f'Bearer {LLM_API_KEY}':
        return Response('unauthorized', 401)
    resp = rq.request(
        request.method, f"{OLLAMA}/{p}",
        data=request.get_data(),
        headers={k: v for k, v in request.headers if k.lower() not in ('host', 'authorization')},
        stream=True, timeout=600)
    return Response(resp.iter_content(chunk_size=8192), status=resp.status_code,
                    content_type=resp.headers.get('Content-Type', 'application/json'))

threading.Thread(target=lambda: app.run(host='0.0.0.0', port=PORT, threaded=True), daemon=True).start()
time.sleep(3)
print("✅ البروكسي شغّال على البورت", PORT)

### 6️⃣ تشغيل Cloudflare Tunnel


In [ ]:
import subprocess, time
cf_log = open("/content/cloudflared.log", "w")
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "run", "--token", CF_TUNNEL_TOKEN],
                           stdout=cf_log, stderr=subprocess.STDOUT)
print("⏳ cloudflared بيتصل...")
time.sleep(10)
print(open('/content/cloudflared.log').read()[-1200:])

### 7️⃣ اختبار الـ endpoint العام


In [ ]:
import requests
try:
    r = requests.post("https://colab-llm.cloudstars.club/v1/chat/completions",
        headers={"Authorization": f"Bearer {LLM_API_KEY}"},
        json={"model": MODEL,
              "messages": [{"role": "user", "content": "قول أهلاً في 3 كلمات"}],
              "max_tokens": 30},
        timeout=120)
    print("HTTP", r.status_code)
    print(r.json()["choices"][0]["message"]["content"] if r.status_code == 200 else r.text[:400])
except Exception as e:
    print("❌", e)

### 8️⃣ ابقّ الجلسة شغّالة (اترك هذه الخلية تعمل)


In [ ]:
import time, datetime
print("🟢 الخدمة على: https://colab-llm.cloudstars.club/v1  (model:", MODEL, ")")
while True:
    a = 'OK' if ol_proc.poll() is None else 'DOWN'
    b = 'OK' if cf_proc.poll() is None else 'DOWN'
    print(f"[{datetime.datetime.now():%H:%M:%S}] ollama={a}  tunnel={b}")
    if a == 'DOWN' or b == 'DOWN':
        print("⚠️ عملية وقفت — راجع اللوج وأعد تشغيل خليتها.")
        break
    time.sleep(60)

---
## 🔌 ربطه مع Hermes / opencode

| الحقل | القيمة |
|------|-------|
| **Base URL** | `https://colab-llm.cloudstars.club/v1` |
| **API Key** | قيمة `LLM_API_KEY` (خلية 3) |
| **Model** | `qwen2.5-coder:7b` |

مع fallback تلقائي لـ DeepSeek Flash لو الـ endpoint وقع.
